# Behavioral Feature Engineering Pipeline

## 1. Module Overview & Strategic Objectives
This notebook focuses on deriving high-signal domain features from the sanitized customer records (`cleaned_ecommerce_data.csv`). Raw telemetry indicators alone often fail to capture subtle multi-factor behaviors. By engineering domain-informed interaction ratios, friction metrics, and behavioral flags, we amplify the predictive signals that distinguish loyal customers from churning ones.

### Engineered Feature Categories:
1. **Engagement vs. Friction Ratios:** Quantify content exploration depth relative to customer support friction.
2. **Checkout & Cart Friction Indicators:** Detect high purchase intent blocked by checkout abandonment.
3. **Monetary & Purchasing Velocity:** Measure annualized spend velocity and purchase recency relative to lifetime value.
4. **Behavioral Interaction Features:** Cross-channel engagement proxies combining app, web, and marketing touchpoints.

### import libraries and load cleaned dataset

In [30]:
import numpy as np
import pandas as pd

# load cleaned dataset form data folder.
df = pd.read_csv("../data/cleaned_ecommerce_data.csv")

df.shape
df.head()

,Age,Gender,Country,City,Membership_Years,Login_Frequency,Session_Duration_Avg,Pages_Per_Session,Cart_Abandonment_Rate,Wishlist_Items,...,Churned,Signup_Quarter,Session_Duration_Avg_is_missing,Pages_Per_Session_is_missing,Social_Media_Engagement_Score_is_missing,Mobile_App_Usage_is_missing,Email_Open_Rate_is_missing,Credit_Balance_is_missing,Days_Since_Last_Purchase_is_missing,Discount_Usage_Rate_is_missing
0,43.0,Male,France,Marseille,2.9,14.0,27.4,6.0,50.6,3.0,...,0,Q1,0,0,0,0,0,0,0,0
1,36.0,Male,UK,Manchester,1.6,15.0,42.7,10.3,37.7,1.0,...,0,Q4,0,0,1,0,0,0,0,0
2,45.0,Female,Canada,Vancouver,2.9,10.0,24.8,1.6,70.9,1.0,...,0,Q4,0,0,1,0,0,0,0,0
3,56.0,Female,USA,New York,2.6,10.0,38.4,14.8,41.7,9.0,...,0,Q1,0,0,0,0,0,0,0,0
4,35.0,Male,India,Delhi,3.1,29.0,51.4,8.4,19.1,9.0,...,0,Q4,0,1,0,0,0,0,0,0


### Behavioral Friction & Engagement Feature Construction

In [31]:
# 1. Total Session Browsing Volume (Time spent * Pages viewed)
# Measures aggregate depth of content exploration per visit
df['Total_Session_Volume'] = np.round(df['Session_Duration_Avg'] * df['Pages_Per_Session'], 2)

# 2. Engagement vs. Friction Ratio
# Compares positive browsing exploration against customer support friction
# Added +1 to denominator to avoid division by zero for users with 0 support calls
df['Engagement_Friction_Ratio'] = np.round(
    df['Total_Session_Volume'] / (df['Customer_Service_Calls'] + 1), 2
)

# 3. High Cart Friction Indicator (Binary Flag)
# Flags high purchase intent blocked by checkout abandonment (>70% drop-off with <10 completed orders)
df['High_Cart_Friction'] = (
    (df['Cart_Abandonment_Rate'] > 70.0) & (df['Total_Purchases'] < 10)
).astype(int)

# 4. Wishlist-to-Purchase Conversion Proxy
# Ratio of completed orders relative to items saved in wishlist
df['Wishlist_Conversion_Ratio'] = np.round(
    df['Total_Purchases'] / (df['Wishlist_Items'] + 1), 3
)

# Inspect sample outputs of engineered friction features
friction_cols = [
    'Total_Session_Volume',
    'Engagement_Friction_Ratio',
    'High_Cart_Friction',
    'Wishlist_Conversion_Ratio'
]
df[friction_cols].head(5)

,Total_Session_Volume,Engagement_Friction_Ratio,High_Cart_Friction,Wishlist_Conversion_Ratio
0,164.40,16.44,0,2.25
1,439.81,54.98,0,9.75
2,39.68,7.94,1,4.55
3,568.32,189.44,0,1.50
4,431.76,215.88,0,3.25


### Monetary Velocity & Inactivity Risk Features

In [32]:
# 1. Annual Monetary Velocity using conditional logic (No artificial constant added)
# Agar Membership_Years > 0 hai toh tenure se divide karein, warna current Lifetime_Value hi assign karein
df['Annual_Monetary_Velocity'] = np.where(
    df['Membership_Years'] > 0,
    np.round(df['Lifetime_Value'] / df['Membership_Years'], 2),
    df['Lifetime_Value']
)

# 2. Spend per Purchase Ratio
# Average spend per order (safe fallback if purchases == 0)
df['Spend_Per_Purchase'] = np.where(
    df['Total_Purchases'] > 0,
    np.round(df['Lifetime_Value'] / df['Total_Purchases'], 2),
    df['Lifetime_Value']
)

# 3. Inactivity Risk Flag (Industry standard: > 90 days without purchase)
df['Is_Dormant_90Days'] = (df['Days_Since_Last_Purchase'] > 90.0).astype(int)

# 4. Multi-Channel Digital Engagement Score
df['Digital_Engagement_Composite'] = np.round(
    (df['Mobile_App_Usage'] * 0.4) + 
    (df['Email_Open_Rate'] * 0.3) + 
    (df['Login_Frequency'] * 0.3), 2
)

# Inspect outputs to verify no NaN or Infinite values
monetary_cols = [
    'Annual_Monetary_Velocity',
    'Spend_Per_Purchase',
    'Is_Dormant_90Days',
    'Digital_Engagement_Composite'
]
df[monetary_cols].head(5)

,Annual_Monetary_Velocity,Spend_Per_Purchase,Is_Dormant_90Days,Digital_Engagement_Composite
0,328.73,105.93,0,17.89
1,667.17,54.74,0,26.66
2,444.74,141.73,0,6.52
3,900.35,156.06,0,27.82
4,981.06,93.58,0,40.23


### Integrity Validation

In [33]:
# 1. Check for any infinite or missing values introduced during feature creation
new_features = [
    'Total_Session_Volume', 'Engagement_Friction_Ratio', 'High_Cart_Friction',
    'Wishlist_Conversion_Ratio', 'Annual_Monetary_Velocity', 'Spend_Per_Purchase',
    'Is_Dormant_90Days', 'Digital_Engagement_Composite'
]

print("=== Engineered Features Sanity Check ===")
null_check = df[new_features].isnull().sum()
inf_check = np.isinf(df[new_features]).sum()

sanity_df = pd.DataFrame({
    'Null_Count': null_check,
    'Infinite_Count': inf_check
})
print(sanity_df)

=== Engineered Features Sanity Check ===
                              Null_Count  Infinite_Count
Total_Session_Volume                   0               0
Engagement_Friction_Ratio              0               0
High_Cart_Friction                     0               0
Wishlist_Conversion_Ratio              0               0
Annual_Monetary_Velocity               0               0
Spend_Per_Purchase                     0               0
Is_Dormant_90Days                      0               0
Digital_Engagement_Composite           0               0


### Precision Standardization (Rounding Float Features to 2 Decimals)

In [34]:
# Identify all float columns in the engineered dataset
float_cols = df.select_dtypes(include=['float64', 'float32']).columns

# Round all continuous numerical features to exactly 2 decimal places
df[float_cols] = df[float_cols].round(2)

# Verify updated precision across key continuous features
preview_cols = [
    'Cart_Abandonment_Rate', 
    'Returns_Rate', 
    'Average_Order_Value', 
    'Annual_Monetary_Velocity', 
    'Spend_Per_Purchase'
]

print(f"Total Float Columns Formatted: {len(float_cols)}")
df[preview_cols].head(5)

Total Float Columns Formatted: 26


,Cart_Abandonment_Rate,Returns_Rate,Average_Order_Value,Annual_Monetary_Velocity,Spend_Per_Purchase
0,50.6,2.0,94.72,328.73,105.93
1,37.7,9.2,82.45,667.17,54.74
2,70.9,11.5,165.52,444.74,141.73
3,41.7,5.4,147.33,900.35,156.06
4,19.1,5.5,141.30,981.06,93.58


### Export dataset

In [35]:
df.to_csv("../data/feature_engineering_dataset.csv", index=False)
print("dataset export successfully!")

dataset export successfully!
